In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

from src.load_db import get_engine
from src.logger import get_logger


# ENGINE + LOGGER

def load_power_bi():
    engine = get_engine()
    logger = get_logger(__name__)
    
    
    # EXECUTE STAR SCHEMA SQL
    
    
    with engine.begin() as con:
    
        path_file = r"C:\Users\user\Documents\Conception d'un Data Warehouse et Tableaux de Bord pour l'Analyse du Marché Immobilier - Darkom.ma\sql\star_schema.sql"
    
        with open(path_file, "r", encoding="utf-8") as f:
            sql_script = f.read()
    
        con.exec_driver_sql(sql_script)
    
    logger.info("Star schema créé avec succès")
    
    
    # READ CLEAN DATA
    
    
    df = pd.read_sql(
        "SELECT * FROM schema_clean_darkom.darkom_clean",
        engine
    )
    
    df.columns = df.columns.str.lower()
    
    logger.info("Données chargées depuis darkom_clean")
    
    
    # FIX DATE TYPE
    
    
    df["date_publication"] = pd.to_datetime(
        df["date_publication"]
    )
    
    
    # CREATE DIM_DATE
    
    
    dim_date = df[
        [
            "date_publication",
            "annee_publication",
            "mois_publication",
            "trimestre_publication"
        ]
    ].drop_duplicates()
    
    dim_date.to_sql(
        con=engine,
        schema="star_schema_darkom",
        name="dim_date",
        if_exists="append",
        index=False
    )
    
    logger.info("dim_date chargée")
    
    # CREATE DIM_LOCATION
    
    
    dim_location = df[
        [
            "quartier",
            "ville"
        ]
    ].drop_duplicates()
    
    dim_location.to_sql(
        con=engine,
        schema="star_schema_darkom",
        name="dim_location",
        if_exists="append",
        index=False
    )
    
    logger.info("dim_location chargée")
    
    
    # CREATE DIM_PROPERTY
    
    
    dim_property = df[
        [
            "titre",
            "type_bien",
            "transaction",
            "categorie_prix",
            "categorie_surface",
            "annee_construction"
        ]
    ].drop_duplicates()
    
    dim_property.to_sql(
        con=engine,
        schema="star_schema_darkom",
        name="dim_property",
        if_exists="append",
        index=False
    )
    
    logger.info("dim_property chargée")
    
    
    # READ DIMENSIONS FROM DB
    
    
    dim_date_db = pd.read_sql(
        "SELECT * FROM star_schema_darkom.dim_date",
        engine
    )
    
    dim_location_db = pd.read_sql(
        "SELECT * FROM star_schema_darkom.dim_location",
        engine
    )
    
    dim_property_db = pd.read_sql(
        "SELECT * FROM star_schema_darkom.dim_property",
        engine
    )
    
    
    # FIX DATE TYPE AGAIN
    
    
    dim_date_db["date_publication"] = pd.to_datetime(
        dim_date_db["date_publication"]
    )
    
    
    # MERGE DATE DIMENSION
    
    
    df = df.merge(
        dim_date_db,
        on=[
            "date_publication",
            "annee_publication",
            "mois_publication",
            "trimestre_publication"
        ],
        how="left"
    )
    
    
    # MERGE LOCATION DIMENSION
    
    df = df.merge(
        dim_location_db,
        on=[
            "quartier",
            "ville"
        ],
        how="left"
    )
    
    # MERGE PROPERTY DIMENSION
    
    df = df.merge(
        dim_property_db,
        on=[
            "titre",
            "type_bien",
            "transaction",
            "categorie_prix",
            "categorie_surface",
            "annee_construction"
        ],
        how="left"
    )
    
    logger.info("Merges terminés")
    
    # CREATE FACT TABLE
    
    fact_darkom = df[
        [
            "annonce_id",
            "date_id",
            "location_id",
            "property_id",
            "prix",
            "surface",
            "prix_m2",
            "age_bien",
            "nb_chambres",
            "nb_salles_bain",
            "etage"
        ]
    ].drop_duplicates()
    
    # LOAD FACT TABLE
    
    fact_darkom.to_sql(
        con=engine,
        schema="star_schema_darkom",
        name="fact_darkom_listings",
        if_exists="append",
        index=False
    )
    
    logger.info("fact_darkom_listing chargée avec succès")
    
    print("Pipeline Star Schema terminé avec succès")

2026-05-21 09:41:10,494 - db - INFO - Database connection created successfully
2026-05-21 09:41:10,575 - __main__ - INFO - Star schema créé avec succès
2026-05-21 09:41:10,586 - __main__ - INFO - Données chargées depuis darkom_clean
2026-05-21 09:41:10,599 - __main__ - INFO - dim_date chargée
2026-05-21 09:41:10,605 - __main__ - INFO - dim_location chargée
2026-05-21 09:41:10,627 - __main__ - INFO - dim_property chargée
2026-05-21 09:41:10,639 - __main__ - INFO - Merges terminés
2026-05-21 09:41:10,682 - __main__ - INFO - fact_darkom_listing chargée avec succès


Pipeline Star Schema terminé avec succès


In [ ]:
import logging
import pandas as pd

from src.load_db import get_engine


from src.logger import get_logger
logger = get_logger(__name__)

logger = logging.getLogger(__name__)


def etl_darkom_clean():

    logger.info("Début ETL Clean Layer")

    engine = get_engine()

    try:

        df = pd.read_sql(
            "SELECT * FROM raw_darkom.staging_darkom",
            engine
        )

        logger.info(f"Données chargées : {df.shape[0]} lignes")

        df.columns = df.columns.str.strip().str.lower()

        before = df.shape[0]

        df.drop_duplicates(
            subset="annonce_id",
            inplace=True
        )

        after = df.shape[0]

        logger.info(
            f"Doublons supprimés : {before - after}"
        )

        df["date_publication"] = pd.to_datetime(
            df["date_publication"],
            errors="coerce"
        )

        numeric_cols = [
            "prix",
            "surface",
            "nb_chambres",
            "nb_salles_bain",
            "etage",
            "annee_construction"
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

        logger.info("Colonnes numériques converties")

        df = df.sort_values("date_publication")

        df["date_publication"] = (
            df["date_publication"].ffill()
        )

        cat_cols = [
            "quartier",
            "type_bien",
            "transaction",
            "ville"
        ]

        for col in cat_cols:

            mode_val = df[col].mode()

            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val[0])

            else:
                df[col] = df[col].fillna("Inconnu")

        logger.info("Valeurs catégorielles traitées")

        for col in numeric_cols:
            df[col] = df[col].fillna(
                df[col].median()
            )

        logger.info("Valeurs numériques manquantes traitées")

        def remove_outliers(df, col):

            before_rows = df.shape[0]

            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)

            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            df = df[
                (df[col] >= lower) &
                (df[col] <= upper)
            ]

            after_rows = df.shape[0]

            logger.info(
                f"Outliers supprimés dans {col} : "
                f"{before_rows - after_rows}"
            )

            return df

        for col in [
            "prix",
            "surface",
            "nb_chambres",
            "nb_salles_bain"
        ]:
            df = remove_outliers(df, col)

        logger.info("Traitement des outliers terminé")

        df["prix_m2"] = (
            df["prix"] /
            df["surface"].replace(0, pd.NA)
        )

        current_year = pd.Timestamp.now().year

        df["age_bien"] = (
            current_year -
            df["annee_construction"]
        )

        df.loc[
            df["age_bien"] < 0,
            "age_bien"
        ] = pd.NA

        logger.info("Feature Engineering terminé")

        def categorie_prix(prix):

            if prix < 500000:
                return "Economique"

            elif prix < 1500000:
                return "Moyen"

            elif prix < 3000000:
                return "Haut Standing"

            else:
                return "Luxe"

        def categorie_surface(surface):

            if surface < 80:
                return "Petit"

            elif surface <= 150:
                return "Moyen"

            else:
                return "Grand"

        df["categorie_prix"] = (
            df["prix"].apply(categorie_prix)
        )

        df["categorie_surface"] = (
            df["surface"].apply(categorie_surface)
        )

        df["annee_publication"] = (
            df["date_publication"].dt.year
        )

        df["mois_publication"] = (
            df["date_publication"].dt.month
        )

        df["trimestre_publication"] = (
            df["date_publication"].dt.quarter
        )

        col_int = [
            "nb_chambres",
            "nb_salles_bain",
            "etage",
            "annee_construction"
        ]

        for col in col_int:
            df[col] = df[col].astype("Int64")

        logger.info("Conversion des types terminée")

        file_path = r"C:\Users\user\Documents\Conception d'un Data Warehouse et Tableaux de Bord pour l'Analyse du Marché Immobilier - Darkom.ma\sql\clean.sql"

        with engine.begin() as con:

            with open(
                file_path,
                "r",
                encoding="utf-8"
            ) as f:

                clean_sql = f.read()

            con.exec_driver_sql(clean_sql)

        logger.info("Script SQL exécuté")

        df.to_sql(
            name="darkom_clean",
            con=engine,
            schema="schema_clean_darkom",
            if_exists="append",
            index=False
        )

        logger.info(
            f"Données chargées dans Clean Layer : "
            f"{df.shape[0]} lignes"
        )

        logger.info("ETL terminé avec succès")

        return df

    except Exception as e:

        logger.error(
            f"Erreur ETL Clean Layer : {e}"
        )

        raise


In [ ]:
import pandas as pd 
from sqlalchemy import create_engine,text 
from src.logger import get_logger 
from src.load_db import create_engine
from src.staging impo